# Data Journalism Project: Mapping and Analyzing Landfills in Serbia
## Investigating Waste Management, Environmental Hazards, and Local Impact

### 1. Introduction and Context
In recent years, waste management has emerged as one of the most critical environmental crises in Serbia. This project aims to investigate the current state of waste disposal using official data from the National Open Data Portal (`data.gov.rs`).

The dataset categorizes waste disposal sites into three distinct types:
* **Sanitary Landfills:** Controlled, legally compliant waste management complexes.
* **Non-Sanitary Landfills (Dumpsites):** Municipally operated but substandard sites.
* **Wild Dumpsites:** Wild, unmonitored waste accumulations.

## Part 1: Sanitary Landfills (Sanitarne Deponije)
Sanitary landfills represent regulated waste management facilities. In this section, we ingest the clean data for sanitary landfills and inspect their longitudinal load.

In [1]:
import pandas as pd

# Load the sanitary landfills dataset
df_sanitary = pd.read_excel("sanitarne_deponije.xlsx")

# Print the exact column names to see the actual layout
print("=== ACTUAL EXCEL COLUMNS ===")
print(df_sanitary.columns.tolist())

=== ACTUAL EXCEL COLUMNS ===
['Санитарна депонија', 'Географска ширина (latitude)', 'Географска дужина (longitude)', 'Година', 'Количина отпада (t)']


In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Generate National Trend Data for Datawrapper (Aggregate waste volume per year)
df_sanitary.groupby('Година')['Количина отпада (t)'].sum().reset_index(name='total_waste_tonnes').to_csv("dw_sanitary_national_trend.csv", index=False)

# 2. Generate Individual Landfill Lines Chart Data (Pivoted structure)
lines_chart = df_sanitary.pivot(index='Година', columns='Санитарна депонија', values='Количина отпада (t)').reset_index()
lines_chart.rename(columns={'Година': 'Year'}).to_csv("dw_sanitary_lines_chart.csv", index=False)

# 3. Filter for the latest available reporting year per individual landfill point
df_latest = df_sanitary.sort_values('Година').drop_duplicates('Санитарна депонија', keep='last').copy()

# 4. Map columns to English nomenclature for standardized spatial processing
map_cols = {
    'Санитарна депонија': 'landfill_name',
    'Година': 'latest_reporting_year',
    'Количина отпада (t)': 'waste_volume_tons',
    'Географска ширина (latitude)': 'latitude',   
    'Географска дужина (longitude)': 'longitude'   
}

df_map = df_latest[list(map_cols.keys())].rename(columns=map_cols).dropna(subset=['latitude', 'longitude'])

# 5. Convert DataFrame to GeoDataFrame and export as a standardized GeoJSON layer
geometry = [Point(xy) for xy in zip(df_map['longitude'], df_map['latitude'])]
gpd.GeoDataFrame(df_map, geometry=geometry, crs="EPSG:4326").to_file("mapbox_sanitary_landfills.geojson", driver='GeoJSON')

print("=== ALL EXPORTS SUCCESSFUL ===")

=== ALL EXPORTS SUCCESSFUL ===


## Part 2: Non-Sanitary Landfills (Nesanitarne Deponije)
Non-sanitary landfills are officially recognized but fail to meet modern technical and environmental standards. We integrate a human-in-the-loop spatial filter created in QGIS (`clean_aftermap.csv`) to resolve nomenclature duplication and establish a clean baseline of 154 unique physical locations.

In [3]:
# 1. Load the raw historical non-sanitary data (Excel) and the QGIS verified layout (CSV)
df_nonsanitary = pd.read_excel("nesanitarne_deponije.xlsx")
clean_aftermap = pd.read_csv("clean_aftermap.csv")

# 2. Standardize text keys to ensure perfect matching
for df, name_col, muni_col in [(df_nonsanitary, 'Naziv', 'Opština'), (clean_aftermap, 'landfill_name', 'municipality')]:
    df['name_key'] = df[name_col].astype(str).str.strip().str.lower()
    df['muni_key'] = df[muni_col].astype(str).str.strip().str.lower()

# 3. Extract unique verified coordinates and overwrite historical entries
coords = clean_aftermap[['name_key', 'muni_key', 'latitude', 'longitude']].drop_duplicates()
df_nonsanitary = df_nonsanitary.drop(columns=['Geografska širina', 'Geografska dužina'], errors='ignore')
df_nonsanitary = df_nonsanitary.merge(coords, on=['name_key', 'muni_key'], how='left')

# 4. Restore original structural headers and remove temporary join keys
df_nonsanitary = df_nonsanitary.rename(columns={'latitude': 'Geografska širina', 'longitude': 'Geografska dužina'})
df_nonsanitary = df_nonsanitary.drop(columns=['name_key', 'muni_key'])

print("=== HISTORICAL DATABASE HARMONIZATION SUCCESSFUL ===")
print(f"Total historical rows retained: {df_nonsanitary.shape[0]}")

=== HISTORICAL DATABASE HARMONIZATION SUCCESSFUL ===
Total historical rows retained: 985


In [4]:
# Calculate multi-year reporting continuity and print execution metrics directly
final_spatial_counts = df_nonsanitary.groupby(['Geografska dužina', 'Geografska širina'])['Godina'].nunique().reset_index(name='year_count')

print(f"=== REFINED GEOSPATIAL SUMMARY ===\nActual unique sites verified: {len(final_spatial_counts)}\n Multi-year profiles: {len(final_spatial_counts[final_spatial_counts['year_count'] > 1])}\n Single-year incidents: {len(final_spatial_counts[final_spatial_counts['year_count'] == 1])}")

=== REFINED GEOSPATIAL SUMMARY ===
Actual unique sites verified: 154
 Multi-year profiles: 150
 Single-year incidents: 4


In [29]:
import pandas as pd
import geopandas as gpd

# 1. Chronological sort using original Serbian columns (proven to exist after harmonization)
df_map_sorted = df_nonsanitary.sort_values(by=['Geografska širina', 'Geografska dužina', 'Godina'])

# 2. Optimized categorization using sets for rapid structural validation
def kategorisi_infrastrukturu(series):
    vals = set(series.dropna().astype(str).str.strip().str.lower())
    if 'da' in vals and 'ne' in vals:
        niz = list(series.dropna().astype(str).str.strip().str.lower())
        return 'Gained' if niz.index('da') > niz.index('ne') else 'Lost'
    if 'da' in vals: return 'Always Had'
    return 'Never Had'

# 3. Vectorized aggregation across spatial groups using exact Serbian headers from Excel
safety_cols = {
    'Ograda oko nesanitarne deponije - smetlišta': 'Fencing',
    'Kapija /rampa na ulazu': 'Gate / Ramp',
    'Čuvarska služba': 'Guard Service'
}

df_safety_status = df_map_sorted.groupby(['Geografska dužina', 'Geografska širina']).agg({
    col: kategorisi_infrastrukturu for col in safety_cols.keys()
}).rename(columns=safety_cols)

# 4. Generate and export the Datawrapper Donut Chart matrix for safety
donut_safety = pd.DataFrame({col: df_safety_status[col].value_counts() for col in safety_cols.values()})
donut_safety = donut_safety.reindex(['Always Had', 'Gained', 'Lost', 'Never Had']).fillna(0).astype(int)
donut_safety.reset_index().rename(columns={'index': 'Status'}).to_csv("donut__data.csv", index=False)

# 5. Isolate latest profiles, calculate composite safety score, and map variables
df_latest_safety = df_map_sorted.drop_duplicates(subset=['Geografska širina', 'Geografska dužina'], keep='last').copy()

# Convert raw Serbian "da/ne" columns into binary integers (1 if 'da', else 0)
df_latest_safety['fencing_num'] = df_latest_safety['Ograda oko nesanitarne deponije - smetlišta'].apply(lambda v: 1 if str(v).strip().lower() == 'da' else 0)
df_latest_safety['gate_num'] = df_latest_safety['Kapija /rampa na ulazu'].apply(lambda v: 1 if str(v).strip().lower() == 'da' else 0)
df_latest_safety['guard_num'] = df_latest_safety['Čuvarska služba'].apply(lambda v: 1 if str(v).strip().lower() == 'da' else 0)

# Create a composite safety score (0 to 3) indicating total number of safety measures present
df_latest_safety['safety_score'] = (df_latest_safety['fencing_num'] + df_latest_safety['gate_num'] + df_latest_safety['guard_num']).fillna(0).astype(int)

# Maintain the dynamic "da/ne" text columns for safety tooltips/popups
for src, target in safety_cols.items():
    col_name = target.lower().replace(' / ', '_').replace(' ', '_')
    df_latest_safety[col_name] = df_latest_safety[src].apply(lambda v: 'da' if str(v).strip().lower() == 'da' else 'ne')

# Map and clean database headers for Mapbox safety integration
mapbox_safety_columns = {
    'Naziv': 'landfill_name', 
    'Opština': 'municipality', 
    'Zauzete katastarske parcele': 'cadastral_parcel',
    'Godina': 'latest_reporting_year', 
    'fencing': 'fencing_x', 
    'gate_ramp': 'gate_x', 
    'guard_service': 'guard_x',
    'safety_score': 'safety_score'
}

df_spatial_safety = df_latest_safety[list(mapbox_safety_columns.keys())].rename(columns=mapbox_safety_columns)

# 6. CRITICAL FIX: Load accurate geometries from QGIS JSON to overwrite bad CSV coordinates
gdf_qgis = gpd.read_file('clean_aftermap_for_qgis.geojson')

# Merge cleaned spatial coordinates with analyzed safety attributes (left join with 0 fill)
gdf_fixed_safety = gdf_qgis.merge(df_spatial_safety, left_on='landfill_name', right_on='landfill_name', how='left')
gdf_fixed_safety['safety_score'] = gdf_fixed_safety['safety_score'].fillna(0).astype(int)

# 7. Export clean spatial dataset for Mapbox integration
gdf_fixed_safety.to_file("mapbox_landfills_safety.geojson", driver='GeoJSON')

print("=== SAFETY EXPORT SUCCESSFUL: QGIS GEOMETRY MASTER APPLIED WITH SAFETY-INDEX ===")

=== SAFETY EXPORT SUCCESSFUL: QGIS GEOMETRY MASTER APPLIED WITH SAFETY-INDEX ===


### Part 2.1: Environmental Hazard & Mitigation Infrastructure
Beyond basic site security, the true environmental risk posed by non-sanitary landfills lies in sub-surface and atmospheric contamination. This section evaluates the presence and chronological implementation of critical ecological protection systems:
* **Gas Extraction (`degassing_system`):** Systems preventing toxic and explosive landfill gas build-up.
* **Leachate Drainage (`leachate_drainage`):** Infrastructure capturing highly toxic liquid runoff.
* **Leachate Treatment (`leachate_treatment`):** Facilities purifying captured liquid waste before it infiltrates local groundwater networks.

In [23]:
import pandas as pd
import geopandas as gpd

# 1. Chronological sort using original Serbian column names
df_map_sorted = df_nonsanitary.sort_values(by=['Geografska širina', 'Geografska dužina', 'Godina'])

# 2. Optimized categorization using sets for rapid structural validation
def kategorisi_infrastrukturu(series):
    vals = set(series.dropna().astype(str).str.strip().str.lower())
    if 'da' in vals and 'ne' in vals:
        niz = list(series.dropna().astype(str).str.strip().str.lower())
        return 'Gained' if niz.index('da') > niz.index('ne') else 'Lost'
    if 'da' in vals: return 'Always Had'
    return 'Never Had'

# 3. Vectorized aggregation across spatial groups (Replaces the slow 'for' loop)
eco_cols = {
    'Sistem za otplinavanje deponijskog gasa': 'Gas Extraction',
    'Drenažni sistem za prikupljanje procednih otpadnih voda': 'Leachate Drainage',
    'Sistem za prečišćavanje procednih voda sa semtlišta': 'Leachate Treatment'
}

df_eco_status = df_map_sorted.groupby(['Geografska dužina', 'Geografska širina']).agg({
    col: kategorisi_infrastrukturu for col in eco_cols.keys()
}).rename(columns=eco_cols)

# 4. Generate and export the Datawrapper Donut Chart matrix
donut_eco = pd.DataFrame({col: df_eco_status[col].value_counts() for col in eco_cols.values()})
donut_eco = donut_eco.reindex(['Always Had', 'Gained', 'Lost', 'Never Had']).fillna(0).astype(int)
donut_eco.reset_index().rename(columns={'index': 'Status'}).to_csv("donut_environmental_data.csv", index=False)

# 5. Isolate latest profiles, calculate composite environmental score, and map binary variables
df_latest_eco = df_map_sorted.drop_duplicates(subset=['Geografska širina', 'Geografska dužina'], keep='last').copy()

# Convert the raw "da/ne" columns into binary integers (1 if 'da', else 0)
df_latest_eco['gas_num'] = df_latest_eco['Sistem za otplinavanje deponijskog gasa'].apply(lambda v: 1 if str(v).strip().lower() == 'da' else 0)
df_latest_eco['drainage_num'] = df_latest_eco['Drenažni sistem za prikupljanje procednih otpadnih voda'].apply(lambda v: 1 if str(v).strip().lower() == 'da' else 0)
df_latest_eco['treatment_num'] = df_latest_eco['Sistem za prečišćavanje procednih voda sa semtlišta'].apply(lambda v: 1 if str(v).strip().lower() == 'da' else 0)

# Create a composite score (0 to 3) indicating the total number of protective eco-systems present
df_latest_eco['environmental_score'] = df_latest_eco['gas_num'] + df_latest_eco['drainage_num'] + df_latest_eco['treatment_num']

# Maintain the dynamic "da/ne" text columns for map tooltips/popups
for src, target in eco_cols.items():
    col_name = target.lower().replace(' ', '_')
    df_latest_eco[col_name] = df_latest_eco[src].apply(lambda v: 'da' if str(v).strip().lower() == 'da' else 'ne')

# Map and clean database headers for Mapbox integration (added the new 'environmental_score')
mapbox_eco_columns = {
    'Naziv': 'landfill_name', 
    'Opština': 'municipality', 
    'Zauzete katastarske parcele': 'cadastral_parcel',
    'Godina': 'latest_reporting_year', 
    'gas_extraction': 'degassing_system', 
    'leachate_drainage': 'leachate_drainage', 
    'leachate_treatment': 'leachate_treatment',
    'environmental_score': 'environmental_score'
}

df_spatial_eco = df_latest_eco[list(mapbox_eco_columns.keys())].rename(columns=mapbox_eco_columns)

# 6. CRITICAL FIX: Load accurate geometries from QGIS JSON to overwrite bad CSV coordinates
gdf_qgis = gpd.read_file('clean_aftermap_for_qgis.geojson')

# Merge cleaned spatial coordinates with analyzed environmental attributes
gdf_fixed_eco = gdf_qgis.merge(df_spatial_eco, left_on='landfill_name', right_on='landfill_name', how='inner')

# 7. Export clean spatial dataset for Mapbox integration (No dirty CSV exports)
gdf_fixed_eco.to_file("mapbox_environmental_hazards.geojson", driver='GeoJSON')

print("=== ENVIRONMENTAL EXPORT SUCCESSFUL: QGIS GEOMETRY MASTER APPLIED WITH ECO-INDEX ===")

=== ENVIRONMENTAL EXPORT SUCCESSFUL: QGIS GEOMETRY MASTER APPLIED WITH ECO-INDEX ===


### Part 2.2.1: Operational Practices & Historical Variations
A critical dimension of managing non-sanitary landfills is the technical method used to dump and process waste on-site. In municipal engineering, disposal techniques follow a strict environmental hierarchy—ranging from structured space management to chaotic, unmanaged piling.

To uncover structural shifts in municipal management, this section tracks whether local authorities upgraded, degraded, or maintained their operational standards over time. Our historical analysis reveals that out of 154 unique physical landfill sites, **58 sites changed their disposal method** over the reporting years, while **96 sites maintained a completely static approach** throughout their history.

In [7]:
# Check unique text variations in 'Način odlaganja otpada'
print("=== UNIQUE WASTE DISPOSAL METHODS ===")
options = df_nonsanitary['Način odlaganja otpada'].astype(str).str.strip().unique()
for option in options:
    print(f"- {option}")

=== UNIQUE WASTE DISPOSAL METHODS ===
- Razvrstavanje otpada uz sabijanje i ravnanje
- Sloj po sloj
- Nekontrolisano odlaganje
- Po kasetama


In [8]:
import pandas as pd
import geopandas as gpd

# 1. Chronological sort using original Serbian column names
df_map_sorted = df_nonsanitary.sort_values(by=['Geografska širina', 'Geografska dužina', 'Godina'])

# Dictionary for clean English translation and mapping
method_mapping = {
    'Po kasetama': 'In Cassettes / Cells',
    'Razvrstavanje otpada uz sabijanje i ravnanje': 'Compacting & Levelling',
    'Sloj po sloj': 'Layer by Layer',
    'Nekontrolisano odlaganje': 'Uncontrolled Dumping'
}

# Translate and clean the main column in our sorted database
df_map_sorted['method_en'] = df_map_sorted['Način odlaganja otpada'].map(method_mapping).fillna('Unreported / Missing')

# --- EXPORT 1: HISTORICAL TREND (100% Stacked Bar) ---
historical_counts = df_map_sorted.groupby(['Godina', 'method_en']).size().unstack(fill_value=0)
historical_trend_pct = historical_counts.div(historical_counts.sum(axis=1), axis=0) * 100
historical_trend_pct.reset_index().rename(columns={'Godina': 'Year'}).to_csv("disposal_historical_trend.csv", index=False)

# --- EXPORT 2: LATEST PROFILE (Donut Chart) ---
df_latest_disposal = df_map_sorted.drop_duplicates(subset=['Geografska širina', 'Geografska dužina'], keep='last').copy()
donut_disposal = df_latest_disposal['method_en'].value_counts().reset_index()
donut_disposal.columns = ['Disposal Method', 'Site Count']
donut_disposal.to_csv("disposal_latest_donut.csv", index=False)

# --- EXPORT 3: ISOLATED SPATIAL DATASETS (New Independent Map) ---
# Select and rename attributes needed for Mapbox tooltips/popups (removed raw coordinates)
spatial_cols = {
    'Naziv': 'landfill_name', 
    'Opština': 'municipality', 
    'Zauzete katastarske parcele': 'cadastral_parcel', 
    'Godina': 'latest_reporting_year',
    'method_en': 'disposal_method'
}

df_spatial_disposal = df_latest_disposal[list(spatial_cols.keys())].rename(columns=spatial_cols)

# 6. CRITICAL FIX: Load accurate geometries from QGIS JSON to overwrite bad CSV coordinates
gdf_qgis = gpd.read_file('clean_aftermap_for_qgis.geojson')

# Merge cleaned spatial coordinates with analyzed disposal attributes
gdf_fixed_disposal = gdf_qgis.merge(df_spatial_disposal, left_on='landfill_name', right_on='landfill_name', how='inner')

# 7. Export clean spatial dataset for Mapbox integration (No dirty CSV exports)
gdf_fixed_disposal.to_file("mapbox_disposal_methods.geojson", driver='GeoJSON')

print("=== ALL INDEPENDENT DISPOSAL EXPORTS SUCCESSFUL ===")
print("Generated: 'disposal_latest_donut.csv', 'disposal_historical_trend.csv' & 'mapbox_disposal_methods.geojson'")

=== ALL INDEPENDENT DISPOSAL EXPORTS SUCCESSFUL ===
Generated: 'disposal_latest_donut.csv', 'disposal_historical_trend.csv' & 'mapbox_disposal_methods.geojson'


### Part 2.3: Leadboards and metrics for non-sanitary landfills 

In [9]:
# 1. Isolate the latest reporting profile per unique non-sanitary site (using verified coordinates)
df_latest_nonsanitary = df_nonsanitary.sort_values(['Geografska širina', 'Geografska dužina', 'Godina']).drop_duplicates(subset=['Geografska širina', 'Geografska dužina'], keep='last')

# 2. Define the exact volume column from your verified columns list
volume_col_ns = 'Prosečne godišnje količine otpada, koji se odlaže na nesanitarnu deponiju - smetlište (t)'

# --- MUNICIPAL LEVEL LEADERBOARD ---
df_mun_nonsanitary = df_latest_nonsanitary.groupby('Opština').agg(
    site_count=('Opština', 'size'),
    total_volume_tons=(volume_col_ns, 'sum')
).reset_index()

print("=" * 60)
print("     STORY METRICS: TOP 5 NON-SANITARY MUNICIPALITIES     ")
print("=" * 60)

print("\nTOP 5 MUNICIPALITIES BY TOTAL WASTE VOLUME (Tons):")
print(df_mun_nonsanitary.sort_values('total_volume_tons', ascending=False)[['Opština', 'total_volume_tons']].head(5).to_string(index=False, header=['Municipality', 'Waste Volume (t)']))

print("\nTOP 5 MUNICIPALITIES BY NUMBER OF ACTIVE NON-SANITARY SITES:")
print(df_mun_nonsanitary.sort_values('site_count', ascending=False)[['Opština', 'site_count']].head(5).to_string(index=False, header=['Municipality', 'Site Count']))


# --- INDIVIDUAL SITE LEVEL LEADERBOARD ---
# Using 'Naziv' for the site name and 'Lokacija (mesto, naselje)' for the granular location
fields_ns = ['Opština', 'Naziv', 'Lokacija (mesto, naselje)', volume_col_ns]

print("\n" + "=" * 80)
print("     STORY METRICS: TOP 5 LARGEST INDIVIDUAL NON-SANITARY LANDFILLS     ")
print("=" * 80)

print("\nTOP 5 INDIVIDUAL SITES BY WASTE VOLUME:")
print(df_latest_nonsanitary.sort_values(volume_col_ns, ascending=False)[fields_ns].head(5).to_string(index=False, header=['Municipality', 'Site Name', 'Location', 'Volume (t)']))
print("\n" + "=" * 80)

     STORY METRICS: TOP 5 NON-SANITARY MUNICIPALITIES     

TOP 5 MUNICIPALITIES BY TOTAL WASTE VOLUME (Tons):
     Municipality Waste Volume (t)
  Novi Sad - grad         366901.0
Kragujevac - grad         226839.0
Beograd-Obrenovac         160000.0
   Niš - Palilula         118661.3
          Pančevo         100000.0

TOP 5 MUNICIPALITIES BY NUMBER OF ACTIVE NON-SANITARY SITES:
Municipality Site Count
       Kovin          8
        Kula          6
    Leskovac          4
       Titel          4
    Alibunar          3

     STORY METRICS: TOP 5 LARGEST INDIVIDUAL NON-SANITARY LANDFILLS     

TOP 5 INDIVIDUAL SITES BY WASTE VOLUME:
     Municipality                         Site Name       Location Volume (t)
  Novi Sad - grad     Gradska deponija u Novom Sadu       Novi Sad   366901.0
Kragujevac - grad                Deponija Jovanovac     Kragujevac   226839.0
   Niš - Palilula Gradska deponija ‚‚Bubanj'' - Niš Niš (Palilula)   118661.3
          Pančevo                    Stara dep

### Part 2.4: Non-sanitary reporting gap

In [10]:
import pandas as pd

# Load the file
df_jls = pd.read_excel('kontakt-podaci-jls.xlsx')

# Clean and display column names to spot any hidden spaces or exact casing
print("=== EXACT COLUMN NAMES IN THE FILE ===")
for i, col in enumerate(df_jls.columns):
    print(f"Index {i}: '{col}'")

=== EXACT COLUMN NAMES IN THE FILE ===
Index 0: 'Назив јединице локалне самоуправе'
Index 1: 'Адреса јединице локалне самоуправе'
Index 2: 'Име и презиме градоначелника/председника општине'
Index 3: 'Контакт телефон градоначелника/председника општине'
Index 4: 'Email адреса градоначелника/председника општине'
Index 5: 'Име и презиме председника скупштине'
Index 6: 'Контакт телефон председника скупштине'
Index 7: 'Email адреса председника скупштине'
Index 8: 'Име и презиме начелника општинске/градске управе'
Index 9: 'Контакт телефон начелника општинске/градске управе'
Index 10: 'Email адреса начелника општинске/градске управе'


In [11]:
import pandas as pd

# 1. Load data and define the Cyrillic-to-Latin dictionary for mapping
df_jls = pd.read_excel('kontakt-podaci-jls.xlsx')

cyr_to_lat_dict = {
    'А': 'A', 'Б': 'B', 'В': 'V', 'Г': 'G', 'Д': 'D', 'Ђ': 'Đ', 'Е': 'E', 'Ж': 'Ž',
    'З': 'Z', 'И': 'I', 'Ј': 'J', 'К': 'K', 'Л': 'L', 'Љ': 'Lj', 'М': 'M', 'Н': 'N',
    'Њ': 'Nj', 'О': 'O', 'П': 'P', 'Р': 'R', 'С': 'S', 'Т': 'T', 'Ћ': 'Ć', 'У': 'U',
    'Ф': 'F', 'Х': 'H', 'Ц': 'C', 'Ч': 'Č', 'Џ': 'Dž', 'Ш': 'Š',
    'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd', 'ђ': 'đ', 'е': 'e', 'ж': 'ž',
    'з': 'z', 'и': 'i', 'ј': 'j', 'к': 'k', 'л': 'l', 'љ': 'lj', 'м': 'm', 'н': 'n',
    'њ': 'nj', 'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't', 'ћ': 'ć', 'у': 'u',
    'ф': 'f', 'х': 'h', 'ц': 'c', 'ч': 'č', 'џ': 'dž', 'ш': 'š'
}

def transliterate(text):
    text = str(text)
    for cyr, lat in cyr_to_lat_dict.items():
        text = text.replace(cyr, lat)
    return text

# 2. Standardize both datasets into clean uppercase sets
jls_column = 'Назив јединице локалне самоуправе'
jls_set = set(df_jls[jls_column].dropna().apply(transliterate).str.strip().str.upper())

# NOTE: Change 'df_nonsanitary' to 'df_wild' when running this in Part 3
reported_set = set(df_nonsanitary['Opština'].dropna().astype(str).str.strip().str.upper())

# 3. Compute symmetric differences directly
missing_in_dataset = sorted(list(jls_set - reported_set))
missing_in_jls = sorted(list(reported_set - jls_set))

# 4. Straightforward output display
print(f"=== SET 1: IN JLS MASTER LIST BUT MISSING IN LANDFILL DATA ({len(missing_in_dataset)}) ===")
for muni in missing_in_dataset:
    print(f"- {muni}")

print(f"\n=== SET 2: IN LANDFILL DATA BUT NOT FOUND IN JLS MASTER LIST ({len(missing_in_jls)}) ===")
for muni in missing_in_jls:
    print(f"- {muni}")

=== SET 1: IN JLS MASTER LIST BUT MISSING IN LANDFILL DATA (31) ===
- ARANĐELOVAC
- BABUŠNICA
- BATOČINA
- BELA PALANKA
- BEOGRAD
- BOJNIK
- CRNA TRAVA
- DESPOTOVAC
- DOLJEVAC
- GADŽIN HAN
- GORNJI MILANOVAC
- IVANJICA
- KIKINDA
- KRAGUJEVAC
- LEBANE
- LJUBOVIJA
- MALI ZVORNIK
- MEDVEĐA
- MIONICA
- NIŠ
- NOVA CRNJA
- NOVI SAD
- PETROVAC NA MLAVI
- RAČA
- SEČANJ
- SREMSKA MITROVICA
- SREMSKI KARLOVCI
- TUTIN
- VELIKA PLANA
- VRBAS
- ŽABARI

=== SET 2: IN LANDFILL DATA BUT NOT FOUND IN JLS MASTER LIST (8) ===
- BEOGRAD-LAZAREVAC
- BEOGRAD-OBRENOVAC
- BEOGRAD-PALILULA
- KRAGUJEVAC - GRAD
- NIŠ - MEDIANA
- NIŠ - PALILULA
- NOVI SAD - GRAD
- PETROVAC


# PART 3: Wild Dumpsites Analysis
This section focuses on data regarding wild dumpsites in Serbia, which differ significantly from official but non-sanitary landfills as they are entirely unregulated and often unmonitored.

In [12]:
import pandas as pd

# Load the wild dumpsites dataset from the verified local Excel file
df_wild = pd.read_excel("divlje_deponije.xlsx")

# Display the initial rows to inspect the raw data structure
df_wild.head()

,Tip deponije,Godina,Naziv preduzeća,PIB,Matični broj,Opština,Mesto,Šifra mesta,Poštanski broj,Ulica i broj,...,Email,Šifra pretežne delatnosti,Opština2,Naselje,Geografska širina,Geografska dužina,Procenjena količina otpada (t),Procenjena površina smetlišta (m2),Koliko je puta čišćen prostor divlje deponije u toku izveštajne godine?,Da li se na istom mestu ponavlja divlje odlaganje otpada?
0,Divlje deponije,2020,Opština Temerin - Opštinska uprava,101869888,8330514,Temerin,"Temerin, Temerin, 804681",804681,21235,Novosadska 326,...,goran.grkovic@gmail.com,8411,Temerin,"Temerin, Temerin, 804681",45.000000,19.00000,700.0,1500.0,1,Da
1,Divlje deponije,2020,JAVNO KOMUNALNO PREDUZEĆE IZVOR VLADIMIRCI,101403449,7347189,Vladimirci,"Vladimirci, Vladimirci, 709808",709808,15225,SVETOG SAVE 25,...,office@vladimirci.org.rs,3600,Vladimirci,"Dragojevac, Vladimirci, 709867",44.675137,19.86050,10.0,40.0,0,Ne
2,Divlje deponije,2020,JAVNO KOMUNALNO PREDUZEĆE IZVOR VLADIMIRCI,101403449,7347189,Vladimirci,"Vladimirci, Vladimirci, 709808",709808,15225,SVETOG SAVE 25,...,office@vladimirci.org.rs,3600,Vladimirci,"Jalovik, Vladimirci, 709891",44.584170,19.84982,10.0,40.0,0,Da
3,Divlje deponije,2020,JAVNO KOMUNALNO PREDUZEĆE IZVOR VLADIMIRCI,101403449,7347189,Vladimirci,"Vladimirci, Vladimirci, 709808",709808,15225,SVETOG SAVE 25,...,office@vladimirci.org.rs,3600,Vladimirci,"Jazovnik, Vladimirci, 709883",44.562370,19.83990,50.0,70.0,0,Da
4,Divlje deponije,2020,JAVNO KOMUNALNO PREDUZEĆE IZVOR VLADIMIRCI,101403449,7347189,Vladimirci,"Vladimirci, Vladimirci, 709808",709808,15225,SVETOG SAVE 25,...,office@vladimirci.org.rs,3600,Vladimirci,"Jazovnik, Vladimirci, 709883",44.573480,19.88981,100.0,150.0,0,Ne


### Part 3.2: Spatial Consolidation and Metric Distribution of Wild Dumpsites
To eliminate historical duplication and capture the true scale of wild dumping, this section isolates the latest available profile for each of the 10,193 unique physical locations. Using these unique sites, we analyze the total footprint of waste volume (tons) and surface area ($m^2$).

Furthermore, we establish baseline distribution ranges for both volume and area sizes. This allows us to categorize dumpsites from micro-littering zones to massive industrial-scale dumping fields, laying the groundwork for a combined scatter plot analysis and an independent Mapbox layer.

In [13]:
import pandas as pd

# 1. Load data and fix spatial coordinate drift (~20m grid snap)
df_wild = pd.read_excel("divlje_deponije.xlsx")

df_wild['lat_snap'] = df_wild['Geografska širina'].round(4)
df_wild['lon_snap'] = df_wild['Geografska dužina'].round(4)

In [14]:
import pandas as pd

# 1. Load data and fix spatial coordinate drift (~20m grid snap)
df_wild = pd.read_excel("divlje_deponije.xlsx")

df_wild['lat_snap'] = df_wild['Geografska širina'].round(4)
df_wild['lon_snap'] = df_wild['Geografska dužina'].round(4)

# 2. Calculate execution metrics on the stabilized spatial grid
total_rows = len(df_wild)
unique_coordinates = df_wild.groupby(['lon_snap', 'lat_snap']).ngroups

print("=== WILD DUMPSITES SPATIAL REPETITION SUMMARY ===")
print(f"Total number of reporting entries in dataset: {total_rows}")
print(f"Number of UNIQUE physical locations (coordinates): {unique_coordinates}")
print(f"Average reports per unique site: {round(total_rows / unique_coordinates, 2)}\n")

# 3. Isolate the latest reporting profile per unique physical location
df_latest_wild = df_wild.sort_values(['lat_snap', 'lon_snap', 'Godina']).drop_duplicates(subset=['lat_snap', 'lon_snap'], keep='last')

# 4. Setup bins, labels and categorize features simultaneously
v_bins, v_labs = [0, 1, 5, 20, 100, 500, float('inf')], ['Up to 1 t', '1-5 t', '5-20 t', '20-100 t', '100-500 t', 'Over 500 t']
a_bins, a_labs = [0, 30, 100, 300, 1000, 5000, float('inf')], ['Less than 30 m2', '30-100 m2', '100-300 m2', '300-1000 m2', '1000-5000 m2', 'Over 5000 m2']

df_latest_wild['v_label'] = pd.cut(df_latest_wild['Procenjena količina otpada (t)'], bins=v_bins, labels=v_labs)
df_latest_wild['a_label'] = pd.cut(df_latest_wild['Procenjena površina smetlišta (m2)'], bins=a_bins, labels=a_labs)
df_latest_wild['v_coord'] = pd.cut(df_latest_wild['Procenjena količina otpada (t)'], bins=v_bins, labels=range(1, 7)).astype(int)
df_latest_wild['a_coord'] = pd.cut(df_latest_wild['Procenjena površina smetlišta (m2)'], bins=a_bins, labels=range(1, 7)).astype(int)

# 5. Aggregate matrix, structure nomenclature and export
matrix = df_latest_wild.groupby(['v_coord', 'a_coord', 'v_label', 'a_label'], observed=False).size().reset_index(name='Dumpsite Count')
matrix.columns = ['Waste Volume Coordinate', 'Surface Area Coordinate', 'Volume Range Label', 'Area Range Label', 'Dumpsite Count']
matrix[['Waste Volume Coordinate', 'Surface Area Coordinate', 'Dumpsite Count', 'Volume Range Label', 'Area Range Label']].to_csv("wild_dumpsites_scatter_matrix.csv", index=False)

print("=== MATRIX SNAPSHOT (TOP 5 BINS FOR SERBIA) ===")
print(matrix.sort_values(by='Dumpsite Count', ascending=False).head(5).to_string(index=False))

=== WILD DUMPSITES SPATIAL REPETITION SUMMARY ===
Total number of reporting entries in dataset: 18196
Number of UNIQUE physical locations (coordinates): 8115
Average reports per unique site: 2.24

=== MATRIX SNAPSHOT (TOP 5 BINS FOR SERBIA) ===
 Waste Volume Coordinate  Surface Area Coordinate Volume Range Label Area Range Label  Dumpsite Count
                       4                        4           20-100 t      300-1000 m2             660
                       3                        3             5-20 t       100-300 m2             574
                       4                        3           20-100 t       100-300 m2             538
                       3                        2             5-20 t        30-100 m2             468
                       3                        4             5-20 t      300-1000 m2             448


### Part 3.3: Generating the Cross-Tabulation Matrix with Custom Ranges
Based on the statistical distribution of the dataset, we have established tailored intervals for both waste volume and surface area. This matrix cross-tabulates the 10,193 unique dumpsites to map out exactly how volume and area correlate, formatted directly for a Datawrapper Scatter Plot.

In [15]:
import geopandas as gpd
from shapely.geometry import Point

# 1. Aggregate metrics by municipality using the clean, snapped locations from the active grid
df_mun_spatial = df_latest_wild.groupby('Opština').agg(
    dumpsite_count=('Opština', 'size'),
    total_surface_area_m2=('Procenjena površina smetlišta (m2)', 'sum'),
    total_waste_volume_tons=('Procenjena količina otpada (t)', 'sum'),
    latitude=('lat_snap', 'mean'),
    longitude=('lon_snap', 'mean')
).reset_index()

# 2. Standardize nomenclature for web mapping platforms
df_mun_spatial.rename(columns={'Opština': 'municipality'}, inplace=True)
df_mun_spatial = df_mun_spatial.dropna(subset=['latitude', 'longitude'])

# 3. Build and export the clean GeoJSON
geometry = [Point(xy) for xy in zip(df_mun_spatial['longitude'], df_mun_spatial['latitude'])]
gpd.GeoDataFrame(df_mun_spatial, geometry=geometry, crs="EPSG:4326").to_file("mapbox_municipality_dumpsites.geojson", driver='GeoJSON')

print("=== MUNICIPALITY GEOJSON EXPORTED SUCCESSFULLY ('mapbox_municipality_dumpsites.geojson') ===")

=== MUNICIPALITY GEOJSON EXPORTED SUCCESSFULLY ('mapbox_municipality_dumpsites.geojson') ===


In [16]:
# 1. Aggregate clean metrics per municipality for reporting text reference
df_mun_data = df_latest_wild.groupby('Opština').agg(
    dumpsite_count=('Opština', 'size'),
    total_surface_area_m2=('Procenjena površina smetlišta (m2)', 'sum'),
    total_waste_volume_tons=('Procenjena količina otpada (t)', 'sum')
).reset_index()

# 2. Output data leaderboards for your story draft
print("=" * 60)
print("     STORY METRICS: TOP 5 MOST CRITICAL MUNICIPALITIES     ")
print("=" * 60)

print("\nTOP 5 BY WILD DUMPSITE COUNT:")
print(df_mun_data.sort_values('dumpsite_count', ascending=False)[['Opština', 'dumpsite_count']].head(5).to_string(index=False, header=['Municipality', 'Dumpsite Count']))

print("\nTOP 5 BY TOTAL WASTE VOLUME (Tons):")
print(df_mun_data.sort_values('total_waste_volume_tons', ascending=False)[['Opština', 'total_waste_volume_tons']].head(5).to_string(index=False, header=['Municipality', 'Waste Volume (t)']))

print("\nTOP 5 BY TOTAL SURFACE AREA (m²):")
print(df_mun_data.sort_values('total_surface_area_m2', ascending=False)[['Opština', 'total_surface_area_m2']].head(5).to_string(index=False, header=['Municipality', 'Surface Area (m²)']))
print("\n" + "=" * 60)

     STORY METRICS: TOP 5 MOST CRITICAL MUNICIPALITIES     

TOP 5 BY WILD DUMPSITE COUNT:
    Municipality Dumpsite Count
        Leskovac            463
       Aleksinac            351
Beograd-Palilula            339
        Kraljevo            292
          Vranje            266

TOP 5 BY TOTAL WASTE VOLUME (Tons):
   Municipality Waste Volume (t)
      Smederevo       2007560.00
       Srbobran       1005170.00
         Sombor        613380.00
        Pančevo        566505.00
Novi Sad - grad        399334.64

TOP 5 BY TOTAL SURFACE AREA (m²):
Municipality Surface Area (m²)
     Pančevo         2418724.0
    Kovačica         1413215.8
    Leskovac         1238225.0
     Negotin         1045600.0
     Žitište          733722.0



In [17]:
# 1. Select fields and extract top individual records from the spatial grid
fields = ['Opština', 'Naselje', 'Procenjena količina otpada (t)', 'Procenjena površina smetlišta (m2)', 'lat_snap', 'lon_snap']

top_ind_volume = df_latest_wild[fields].sort_values('Procenjena količina otpada (t)', ascending=False).head(5).copy()
top_ind_area = df_latest_wild[fields].sort_values('Procenjena površina smetlišta (m2)', ascending=False).head(5).copy()

# 2. Print results for analytical text support
print("=" * 80)
print("     STORY METRICS: TOP 5 LARGEST INDIVIDUAL WILD DUMPSITES     ")
print("=" * 80)

print("\nTOP 5 INDIVIDUAL DUMPSITES BY WASTE VOLUME:")
print(top_ind_volume.to_string(index=False, header=['Municipality', 'Settlement', 'Volume (t)', 'Area (m²)', 'Lat', 'Lon']))

print("\nTOP 5 INDIVIDUAL DUMPSITES BY SURFACE AREA:")
print(top_ind_area.to_string(index=False, header=['Municipality', 'Settlement', 'Volume (t)', 'Area (m²)', 'Lat', 'Lon']))
print("\n" + "=" * 80)

# 3. Combine both dataframes and deduplicate to catch overlapping extreme sites
df_top_maps = pd.concat([top_ind_volume, top_ind_area]).drop_duplicates(subset=['lat_snap', 'lon_snap'])

# 4. Standardize nomenclature for web mapping layers
df_top_maps.rename(columns={
    'Opština': 'municipality',
    'Naselje': 'settlement',
    'Procenjena količina otpada (t)': 'waste_volume_tons',
    'Procenjena površina smetlišta (m2)': 'surface_area_m2',
    'lat_snap': 'latitude',
    'lon_snap': 'longitude'
}, inplace=True)

# 5. Build and export GeoJSON with the required English filename
geometry = [Point(xy) for xy in zip(df_top_maps['longitude'], df_top_maps['latitude'])]
gpd.GeoDataFrame(df_top_maps, geometry=geometry, crs="EPSG:4326").to_file("top_wild_landfills.geojson", driver='GeoJSON')

print("=== GEOJSON MAP EXPORTED SUCCESSFULLY ('top_wild_landfills.geojson') ===")

     STORY METRICS: TOP 5 LARGEST INDIVIDUAL WILD DUMPSITES     

TOP 5 INDIVIDUAL DUMPSITES BY WASTE VOLUME:
Municipality                  Settlement Volume (t) Area (m²)     Lat     Lon
   Smederevo  Radinac, Smederevo, 740454  2000000.0   20000.0 44.5838 20.9693
    Srbobran    Nadalj, Srbobran, 804037  1000000.0    5000.0 45.3049 19.5447
      Sombor      Sombor, Sombor, 803979   433680.0   55600.0 45.7580 19.1019
    Kovačica Debeljača, Kovačica, 802239   190000.0   21327.0 45.0682 20.5874
     Pančevo    Ivanovo, Pančevo, 803090   130000.0  177086.0 44.7353 20.6846

TOP 5 INDIVIDUAL DUMPSITES BY SURFACE AREA:
Municipality                          Settlement Volume (t) Area (m²)     Lat     Lon
    Kovačica           Crepaja, Kovačica, 802301     3847.0  668968.0 45.0075 20.6246
     Pančevo Banatsko Novo Selo, Pančevo, 803065    40000.0  562452.0 44.9870 20.7946
     Pančevo             Jabuka, Pančevo, 803103      400.0  538069.0 44.9503 20.5670
     Pančevo           Kačarevo, 

### Part 3.4: Wild reporting gap

In [18]:

import pandas as pd

# 1. Load the official JLS master list and handle Cyrillic-to-Latin translation
df_jls = pd.read_excel('kontakt-podaci-jls.xlsx')

cyr_to_lat_dict = {
    'А': 'A', 'Б': 'B', 'В': 'V', 'Г': 'G', 'Д': 'D', 'Ђ': 'Đ', 'Е': 'E', 'Ж': 'Ž',
    'З': 'Z', 'И': 'I', 'Ј': 'J', 'К': 'K', 'Л': 'L', 'Љ': 'Lj', 'М': 'M', 'Н': 'N',
    'Њ': 'Nj', 'О': 'O', 'П': 'P', 'Р': 'R', 'С': 'S', 'Т': 'T', 'Ћ': 'Ć', 'У': 'U',
    'Ф': 'F', 'Х': 'H', 'Ц': 'C', 'Ч': 'Č', 'Џ': 'Dž', 'Ш': 'Š',
    'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd', 'ђ': 'đ', 'е': 'e', 'ж': 'ž',
    'з': 'z', 'и': 'i', 'ј': 'j', 'к': 'k', 'л': 'l', 'љ': 'lj', 'м': 'm', 'н': 'n',
    'њ': 'nj', 'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't', 'ћ': 'ć', 'у': 'u',
    'ф': 'f', 'х': 'h', 'ц': 'c', 'ч': 'č', 'џ': 'dž', 'ш': 'š'
}

def transliterate(text):
    text = str(text)
    for cyr, lat in cyr_to_lat_dict.items():
        text = text.replace(cyr, lat)
    return text

# 2. Extract standardized uppercase sets from both data sources
jls_column = 'Назив јединице локалне самоуправе'
jls_set = set(df_jls[jls_column].dropna().apply(transliterate).str.strip().str.upper())
reported_wild_set = set(df_latest_wild['Opština'].dropna().astype(str).str.strip().str.upper())

# 3. Compute structural differences directly
missing_in_wild_dataset = sorted(list(jls_set - reported_wild_set))
missing_in_jls_master = sorted(list(reported_wild_set - jls_set))

# 4. Display the discrepancy report
print(f"=== SET 1: IN JLS MASTER LIST BUT MISSING IN WILD DUMPSITE DATA ({len(missing_in_wild_dataset)}) ===")
for muni in missing_in_wild_dataset:
    print(f"- {muni}")

print(f"\n=== SET 2: IN WILD DUMPSITE DATA BUT NOT FOUND IN JLS MASTER LIST ({len(missing_in_jls_master)}) ===")
for muni in missing_in_jls_master:
    print(f"- {muni}")

=== SET 1: IN JLS MASTER LIST BUT MISSING IN WILD DUMPSITE DATA (10) ===
- BAČKI PETROVAC
- BEOGRAD
- CRNA TRAVA
- KRAGUJEVAC
- NIŠ
- NOVI SAD
- PETROVAC NA MLAVI
- SREMSKI KARLOVCI
- TUTIN
- ČAJETINA

=== SET 2: IN WILD DUMPSITE DATA BUT NOT FOUND IN JLS MASTER LIST (13) ===
- BEOGRAD-BARAJEVO
- BEOGRAD-LAZAREVAC
- BEOGRAD-MLADENOVAC
- BEOGRAD-OBRENOVAC
- BEOGRAD-PALILULA
- BEOGRAD-SAVSKI VENAC
- BEOGRAD-SURČIN
- BEOGRAD-ZVEZDARA
- KRAGUJEVAC - GRAD
- NIŠ - MEDIANA
- NIŠ - PALILULA
- NOVI SAD - GRAD
- PETROVAC


## 4. Reporting Dashboard: Integrated Analytical Key-Metrics
This centralized control panel aggregates computed metrics across sanitary, non-sanitary, and wild dumpsite reporting datasets to provide structured, production-ready data insights for journalistic drafting.

In [19]:
print("=== INTRO METRICS BY LANDFILL TYPE ===")
# 1. Sanitary Landfills
san_count = len(df_map)
san_tons = df_map['waste_volume_tons'].sum()  # Koristi mapiranu kolonu iz Ćelije 2
print(f" * Sanitary Landfills: {san_count} regional facilities | Total Waste: {san_tons:,.2f} t")

# 2. Non-Sanitary Landfills
ns_count = len(df_latest_nonsanitary)
ns_vol_col = 'Prosečne godišnje količine otpada, koji se odlaže na nesanitarnu deponiju - smetlište (t)'
ns_tons = df_latest_nonsanitary[ns_vol_col].sum()
print(f" * Non-Sanitary Landfills: {ns_count} official sites | Total Annual Waste: {ns_tons:,.2f} t")

# 3. Wild Dumpsites
wild_unique = df_wild.groupby(['lon_snap', 'lat_snap']).ngroups
df_wild_unique = df_wild.drop_duplicates(subset=['lon_snap', 'lat_snap'])

wild_tons = df_wild_unique['Procenjena količina otpada (t)'].sum()
wild_area_m2 = df_wild_unique['Procenjena površina smetlišta (m2)'].sum()
wild_area_ha = wild_area_m2 / 10000

print(f" * Wild Dumpsites: {wild_unique} unique locations | Total Waste: {wild_tons:,.2f} t | Total Area: {wild_area_m2:,.2f} m² (~{wild_area_ha:,.2f} ha)")

# --- 1. SANITARY LANDFILLS ---
print("\n--- 1. SANITARY LANDFILL REGISTRY PROFILE ---")
if 'df_map' in locals() or 'df_map' in globals():
    print(df_map[['landfill_name', 'latest_reporting_year', 'waste_volume_tons']].sort_values(by='waste_volume_tons', ascending=False).to_string(index=False))

# --- 2. SAFETY & SECURITY ---
print("\n--- 2. NON-SANITARY LANDFILLS: SECURITY MATRIX ---")
if 'df_infra_status' in locals() or 'df_infra_status' in globals():
    for col in ['Fencing', 'Gate / Ramp', 'Guard Service']:
        counts = df_infra_status[col].value_counts()
        print(f" * {col} -> Active: {counts.get('Always Had', 0) + counts.get('Gained', 0)} | Always: {counts.get('Always Had', 0)} | Gained: {counts.get('Gained', 0)} | Lost: {counts.get('Lost', 0)} | Never: {counts.get('Never Had', 0)}")

# --- 3. ENVIRONMENTAL HAZARDS ---
print("\n--- 3. NON-SANITARY LANDFILLS: ECO HAZARDS ---")
if 'df_eco_status' in locals() or 'df_eco_status' in globals():
    for col in ['Gas Extraction', 'Leachate Drainage', 'Leachate Treatment']:
        counts = df_eco_status[col].value_counts()
        print(f" * {col} -> Active: {counts.get('Always Had', 0) + counts.get('Gained', 0)} | Always: {counts.get('Always Had', 0)} | Gained: {counts.get('Gained', 0)} | Lost: {counts.get('Lost', 0)} | Never: {counts.get('Never Had', 0)}")

# --- 4. OPERATIONAL METHODS ---
print("\n--- 4. NON-SANITARY LANDFILLS: OPERATIONAL METHOD STRUCTURE ---")
if 'df_latest_disposal' in locals() or 'df_latest_disposal' in globals():
    print(df_latest_disposal['method_en'].value_counts().to_string())

# --- 5. WILD DUMPSITES MUNICIPAL LEADERS ---
print("\n--- 5. WILD DUMPSITES MUNICIPAL LEADERS ---")
if 'df_mun_data' in locals() or 'df_mun_data' in globals():
    print(df_mun_data.sort_values('dumpsite_count', ascending=False)[['Opština', 'dumpsite_count']].head(5).to_string(index=False))

# --- 6. TOP INDIVIDUAL WILD DUMPSITES BY VOLUME ---
print("\n--- 6. TOP INDIVIDUAL WILD DUMPSITES BY VOLUME ---")
if 'top_ind_volume' in locals() or 'top_ind_volume' in globals():
    print(top_ind_volume.to_string(index=False))

# --- 7. DISCREPANCY & SPATIAL MATRIX SUMMARY ---
print("\n--- 7. JLS DISCREPANCY & SCATTER MATRIX SUMMARY ---")
if 'df_wild' in locals() or 'df_wild' in globals():
    print(f" Total entries: {len(df_wild)} | Unique coordinates (snapped): {df_wild.groupby(['lon_snap', 'lat_snap']).ngroups} | Replication rate: {round(len(df_wild) / df_wild.groupby(['lon_snap', 'lat_snap']).ngroups, 2)}")
if 'matrix' in locals() or 'matrix' in globals():
    print("\nTop 5 matrix bins by dumpsite concentration:")
    print(matrix.sort_values(by='Dumpsite Count', ascending=False).head(5).to_string(index=False))

print("-" * 72)

=== INTRO METRICS BY LANDFILL TYPE ===
 * Sanitary Landfills: 12 regional facilities | Total Waste: 1,109,461.00 t
 * Non-Sanitary Landfills: 154 official sites | Total Annual Waste: 2,342,620.21 t
 * Wild Dumpsites: 8115 unique locations | Total Waste: 8,395,301.68 t | Total Area: 24,614,548.62 m² (~2,461.45 ha)

--- 1. SANITARY LANDFILL REGISTRY PROFILE ---
                 landfill_name  latest_reporting_year  waste_volume_tons
          РСД „Винча”, Београд                   2024             569968
          РСД „Гигош“ Јагодина                   2024              97846
                  РСД Суботица                   2024              86916
  РСД „Жељковац – Д2“ Лесковац                   2024              85060
РСД „Јарак“ Сремска Митровица                    2024              80839
                  РСД Панчево                    2024              47462
            РСД „Дубоко“ Ужице                   2024              35328
           РСД „Врбак“ Лапово                    2024 

### 4.2. Enhanced Analytical Insights: Operational Longevity, Remediation Timeline, and Flood Hazard Risks
This section expands the analysis of non-sanitary landfills by investigating their official classifications, active operational statuses, and potential exposure to environmental hazards. Furthermore, it extracts the chronological timeline of approved closure and remediation projects to reconstruct local government interventions over time.

In [20]:
# 1. Legal & Operational Categorization
print("\n[1] Landfill Classification (Nesanitarna deponija - smetlište je):")
print(df_latest_nonsanitary['Nesanitarna deponija - smetlište je:'].value_counts())

print("\n[2] Operational Status (Operativni status):")
print(df_latest_nonsanitary['Operativni status nesanitarne deponije - smetlišta'].value_counts())

print("\n[3] Active Disposal (Da li se i dalje odlaže otpad?):")
print(df_latest_nonsanitary['Da li se na nesanitarnoj deponiji - smetlištu i dalje odlaže otpad?'].value_counts())

# 2. Environmental Risk: Flood Zones
print("\n[4] Flood Zone Risk (Nalazi se u poplavnom području?):")
print(df_latest_nonsanitary['Da li se nesanitarna deponija - smetlište ili njen deo nalazi u poplavnom području?'].value_counts())

# 3. Remediation Projects Analysis
print("\n[5] Remediation Project Status (Izrađen Projekat sanacije?):")
project_counts = df_latest_nonsanitary['Da li je za nesanitarno smetlište izrađen Projekat sanacije, zatvaranja i rekultivacije?'].value_counts()
print(project_counts)

# 4. Timeline Data Generation for Lines Chart (Chronology of Projects)
print("\n[6] Remediation Projects Timeline (Ready for Lines Chart):")
# Filter out non-numeric or missing years, convert to integer
df_latest_nonsanitary['project_year_clean'] = pd.to_numeric(df_latest_nonsanitary['Koje godine?'], errors='coerce')
df_projects_timeline = df_latest_nonsanitary[df_latest_nonsanitary['project_year_clean'].notna()].groupby('project_year_clean').size().reset_index(name='projects_count')
df_projects_timeline['project_year_clean'] = df_projects_timeline['project_year_clean'].astype(int)

# Sort chronologically
df_projects_timeline = df_projects_timeline.sort_values('project_year_clean')
print(df_projects_timeline.to_string(index=False, header=['Year', 'Number of Projects Approved']))

# Export to CSV for Datawrapper Lines Chart
df_projects_timeline.to_csv("dw_remediation_projects_timeline.csv", index=False)
print("\n--> Exported timeline to 'dw_remediation_projects_timeline.csv'")


[1] Landfill Classification (Nesanitarna deponija - smetlište je):
Nesanitarna deponija - smetlište je:
Nesanitarna deponija - smetlište, koja se i dalje koristi i fazno zatvara    113
Nesanitarna deponija - smetlište koje se trajno zatvara                       41
Name: count, dtype: int64

[2] Operational Status (Operativni status):
Operativni status nesanitarne deponije - smetlišta
Aktivno      122
Neaktivno     32
Name: count, dtype: int64

[3] Active Disposal (Da li se i dalje odlaže otpad?):
Da li se na nesanitarnoj deponiji - smetlištu i dalje odlaže otpad?
Da            113
Ne             26
Privremeno     15
Name: count, dtype: int64

[4] Flood Zone Risk (Nalazi se u poplavnom području?):
Da li se nesanitarna deponija - smetlište ili njen deo nalazi u poplavnom području?
Ne    124
Da     30
Name: count, dtype: int64

[5] Remediation Project Status (Izrađen Projekat sanacije?):
Da li je za nesanitarno smetlište izrađen Projekat sanacije, zatvaranja i rekultivacije?
Da    93
Ne

### 4.3. Top 5 largest non-sanitary

In [21]:
print("TOP 5 LARGEST INDIVIDUAL NON-SANITARY LANDFILLS")
if 'df_latest_nonsanitary' in locals() or 'df_latest_nonsanitary' in globals():
    volume_col_ns = 'Prosečne godišnje količine otpada, koji se odlaže na nesanitarnu deponiju - smetlište (t)'
    top_5_sites = df_latest_nonsanitary.sort_values(volume_col_ns, ascending=False).head(5)
    
    for idx, row in top_5_sites.iterrows():
        print(f"\n★ SITE NAME: {row['Naziv']} ({row['Opština']})")
        print(f"  * Annual Waste Load: {row[volume_col_ns]:,.2f} t")
        print(f"  * Granular Location: {row['Lokacija (mesto, naselje)']}")
        print(f"  * Operational Status: {row['Operativni status nesanitarne deponije - smetlišta']}")
        print(f"  * Still Receiving Waste: {row['Da li se na nesanitarnoj deponiji - smetlištu i dalje odlaže otpad?']}")
        print(f"  * Flood Zone Risk: {row['Da li se nesanitarna deponija - smetlište ili njen deo nalazi u poplavnom području?']}")
        
        has_project = row['Da li je za nesanitarno smetlište izrađen Projekat sanacije, zatvaranja i rekultivacije?']
        proj_year = row['Koje godine?'] if pd.notna(row['Koje godine?']) else "N/A"
        print(f"  * Closure/Remediation Project: {has_project} (Year: {proj_year})")
        
        print(f"  * Security: Fence: {row['Ograda oko nesanitarne deponije - smetlišta']} | Gate/Ramp: {row['Kapija /rampa na ulazu']} | Guards: {row['Čuvarska služba']}")
        print(f"  * Eco Controls: Degassing: {row['Sistem za otplinavanje deponijskog gasa']} | Leachate Drainage: {row['Drenažni sistem za prikupljanje procednih otpadnih voda']} | Treatment: {row['Sistem za prečišćavanje procednih voda sa semtlišta']}")
else:
    print(" [Notice] Run non-sanitary data load to generate detailed top 5 profiles.")

TOP 5 LARGEST INDIVIDUAL NON-SANITARY LANDFILLS

★ SITE NAME: Gradska deponija u Novom Sadu (Novi Sad - grad)
  * Annual Waste Load: 366,901.00 t
  * Granular Location: Novi Sad
  * Operational Status: Aktivno
  * Still Receiving Waste: Da
  * Flood Zone Risk: Ne
  * Closure/Remediation Project: Da (Year: 2020.0)
  * Security: Fence: Da | Gate/Ramp: Da | Guards: Da
  * Eco Controls: Degassing: Da | Leachate Drainage: Ne | Treatment: Ne

★ SITE NAME: Deponija Jovanovac (Kragujevac - grad)
  * Annual Waste Load: 226,839.00 t
  * Granular Location: Kragujevac
  * Operational Status: Aktivno
  * Still Receiving Waste: Da
  * Flood Zone Risk: Ne
  * Closure/Remediation Project: Da (Year: 2019.0)
  * Security: Fence: Da | Gate/Ramp: Da | Guards: Da
  * Eco Controls: Degassing: Ne | Leachate Drainage: Ne | Treatment: Ne

★ SITE NAME: Gradska deponija ‚‚Bubanj'' - Niš (Niš - Palilula)
  * Annual Waste Load: 118,661.30 t
  * Granular Location: Niš (Palilula)
  * Operational Status: Aktivno
  * 

In [22]:
from shapely.geometry import Point

# 1. Drop rows with missing coordinates and create a clean copy for spatial analysis
df_spatial_nonsanitary = df_latest_nonsanitary.dropna(
    subset=["Geografska širina", "Geografska dužina"]
).copy()

# 2. Rename columns to clean, standardized English attributes
df_spatial_nonsanitary = df_spatial_nonsanitary.rename(
    columns={
        "Naziv nesanitarne deponije - smetlišta": "landfill_name",
        "Opština": "municipality",
        "Da li se na nesanitarnoj deponiji - smetlištu i dalje odlaže otpad?": "active_disposal",
        "Operativni status nesanitarne deponije - smetlišta": "operational_status",
        "Geografska širina": "latitude",
        "Geografska dužina": "longitude",
    }
)

# 3. Generate geometric Point features using WGS84 projection (EPSG:4326)
geometry_nonsanitary = [
    Point(xy)
    for xy in zip(
        df_spatial_nonsanitary["longitude"], df_spatial_nonsanitary["latitude"]
    )
]
gdf_nonsanitary = gpd.GeoDataFrame(
    df_spatial_nonsanitary, geometry=geometry_nonsanitary, crs="EPSG:4326"
)

# 4. Export the spatial dataframe to an independent GeoJSON file for Mapbox integration
output_filename = "mapbox_nonsanitary_active_status.geojson"
gdf_nonsanitary.to_file(output_filename, driver="GeoJSON")

print("=== NONSANITARY LANDFILLS SPATIAL EXPORT SUCCESSFUL ===")
print(f"Generated independent map layer: '{output_filename}'")

=== NONSANITARY LANDFILLS SPATIAL EXPORT SUCCESSFUL ===
Generated independent map layer: 'mapbox_nonsanitary_active_status.geojson'
